# Introduction to fMRI data in python

---



[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO/blob/main/Borriero_1_Neuroimaging_intro.ipynb)

In [ ]:
from google.colab import auth; auth.authenticate_user()
from google.colab import drive; drive.mount('/content/drive')

 # Import

In [ ]:
# Install the required libraries
! pip install nibabel nilearn nitime

In [ ]:
import numpy as np
import nibabel as nib # For data import and handling
import nilearn as nil # For plotting and analysis
from nilearn import plotting
import nitime # For time series analysis
import matplotlib.pyplot as plt
from nilearn import plotting, surface, datasets
import pandas as pd
from nilearn.image import smooth_img
from nilearn.signal import clean
from tqdm import tqdm


 # Download some files

In [ ]:
# A wrapper method to download files

import os, subprocess, shlex
import os
import subprocess

def download_from_drive(file_keyword, base_folder="."):
    file_name, file_ID = files_dict[file_keyword]
    DEST_PATH = os.path.join(base_folder, file_name)
    if not os.path.exists(DEST_PATH):
        URL = f"https://drive.usercontent.google.com/download?id={file_ID}&confirm=t"
        cmd = ["wget", "-q", "--show-progress", "--progress=bar:force:noscroll",
            "--no-check-certificate", "-O", DEST_PATH, URL]
        ret = subprocess.call(cmd)  # shell=False by default
        if ret != 0:
            raise RuntimeError("Download failed. Check sharing settings or FILE_ID.")
        else:
            print("Downloaded " + file_name + "!")
    else:
        print(file_name + " already present:", DEST_PATH)


In [ ]:
files_dict = {'glasser_labels' : ('glasser_labels.csv','12LumqrRkp1rUAJGg8h3tOIwWubhyyoqY'),
              'glasser_atlas':   ('glasser_atlas.nii.gz','1GeB5pi25SRnyit8s5siDUCFFmJfsB3Yw'),
              'schaefer_labels': ('schaefer_labels.csv', '1Lgh4X9hud2xXGxYEWADFxfMe81P_tTfS'),
              'schaefer_atlas':  ('schaefer_atlas.nii.gz', '1acSTf6m9TyJxm2ruTcwd9eC8PNGfG1PX'),
              'subj_zero_rest':  ('subj_zero_rest.nii.gz', '1XXhVrMymlaBE0ekbgbHoH8fVYteWFGIa')}

In [ ]:
for key in files_dict.keys():
  download_from_drive(key)

# Load fMRI data

In [ ]:
# Load a fMRI volume from a file
"""""
The fMRI data are provided in the so-called nifti image format, identified by the extension .nii.
The extension .gz means that the file is compressed.
Inside the .nii.gz file there is the fMRI volume and a header file

"""""
main_path = '/content/'
file_name = 'subj_zero_rest.nii.gz'
data_ = nib.load(main_path+file_name) # Raw data in nifti image format


In [ ]:
data = data_.get_fdata() # To extract the fMRI in numpy format
header = data_.header
affine = data_.affine

In [ ]:
print(data.shape)

In [ ]:
print(header) # It contains general information about data and acquisition

In [ ]:
affine # It contains spatial and temporal unit of the data

In [ ]:
# Get TR (time repetition)
TR = header.get_zooms()[3]
print(f"TR = {TR} seconds")

In [ ]:
data.shape # The data are shaped (x,y,z,t); x, left to right; y, posterior to anterior; z, inferior to superior

# A first look to fMRI data (first plots)

In [ ]:
# Plot a single time series
fig = plt.figure(figsize=(6,3))
plt.plot(data[13,30,42]);

In [ ]:
# Plot a histogram of all the values different from zero at a given time point
t = 40
plt.hist(data[:,:,:,t][data[:,:,:,t]>0].flatten(), bins=20);

In [ ]:
# Plot some slices with matplotlib
"""""
Name of the cuts:
- fixed x: Saggital, view from the side of the brain
- fixed y: Coronal, view from the front of the brain
- fixed z: Axial, view from the top of the brain

"""""
t = 40
slice = data[:,20,:,t]
plt.imshow(slice)

In [ ]:
### Plot some slices with nibabel
### The plot_stat_map() function (and all the other plotting methods from nilearn) needs the fmri image in nifti format
# Chose a time slice
t = 40
slice = data[:,:,:,t]
slice_nifti = nib.Nifti1Image(slice, affine) # The numpy slice as to be convered back to nifti format
# plotting.plot_stat_map(slice_nifti, display_mode='ortho')
plotting.plot_stat_map(slice_nifti, cut_coords=(10,20,30), display_mode='ortho')

In [ ]:
plotting.plot_stat_map(slice_nifti, cut_coords=(10,20,30), display_mode='mosaic')

# Many kinds of plotting style

In [ ]:
# Glass brain
plotting.plot_glass_brain(slice_nifti)

In [ ]:
### Projection over cortical surface ###
fsaverage = datasets.fetch_surf_fsaverage()
texture_left = surface.vol_to_surf(slice_nifti, fsaverage.pial_left)
texture_right = surface.vol_to_surf(slice_nifti, fsaverage.pial_right)

thresh = 0.2 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
fig.suptitle('Left Primary Motor Cortex FC', x=0.51, y=1)
plt.tight_layout()
plotting.show()

# Atlas

In [ ]:
main_path = '/content/'
file_name = 'glasser_atlas.nii.gz'
file_name_labels = 'glasser_labels.csv'

# path = main_path + 'schaefer_plus_TIAN_SC_LGN_SLEEPresamp.nii.gz'
# path_lab = main_path + 'schaefer_plus_TIAN_SC_LGN.csv'

atlas_ = nib.load(main_path+file_name)
atlas = atlas_.get_fdata()
atlas_header = atlas_.header
atlas_affine = atlas_.affine
atlas_labels = pd.read_csv(main_path+file_name_labels, delimiter=',')
print(atlas.shape)

In [ ]:
# Plot the atlas
plotting.plot_roi(atlas_, title='glasser', display_mode='ortho')

In [ ]:
atlas_labels

In [ ]:
atlas_labels['cortex'].unique()

# Some preprocessing?

1. Motion correction (via FSL or similar)
2. Spatial smoothing
3. Create Brain mask to exclude non-brain voxels
4. Temporal filtering
5. Registration and normalization according to a standard space (e.g. MNI)

In [ ]:
path = '/content/subj_zero_rest.nii.gz'
data_ = nib.load(path)
data = data_.get_fdata()

In [ ]:
# 2. Spatial smoothing
from nilearn.image import smooth_img
smoothed_data_ = smooth_img(data_, fwhm=6)  # FWHM in mm
smoothed_data = smoothed_data_.get_fdata()

# Visualize some voxels
x = 40
y = 30
z = 20
voxel_ts = data[x, y, z, :] # time series of the single voxel
smooth_voxel_ts = smoothed_data[x, y, z, :]
fig,axs = plt.subplots(1,2, figsize=(9,3))
axs[0].plot(smooth_voxel_ts, alpha=1)
axs[0].set_title('smoothed')
axs[1].plot(voxel_ts, alpha=1)
axs[1].set_title('unsmoothed')

In [ ]:
# 3. Create a brain mask
# Dummy masking
brain_mask = np.zeros(data.shape[0:3])
brain_mask[np.where(data.mean(axis=-1)>=20)] =1 # Take all the voxels that are on average greater than zero over time
mask_cord = np.where(brain_mask==1) # Coordinates of all the voxel within the brain mask

slice = brain_mask[20,:,:]
plt.imshow(slice)

In [ ]:
# 3. Create a brain mask
# Atlas masking

# Define a mask from the atlas
brain_mask = np.zeros(atlas.shape)
brain_mask[np.where(atlas!=0)]=1

slice = brain_mask[10,:,:]
plt.imshow(slice)

In [ ]:
# 3. Create a brain mask
# nilearn masking
from nilearn.masking import compute_epi_mask, apply_mask
brain_mask_ = compute_epi_mask(data_)
brain_mask = brain_mask_.get_fdata()

slice = brain_mask[20,:,:]
plt.imshow(slice)

masked_data = apply_mask(data, brain_mask_)

In [ ]:
# 4. Time filtering
from nitime.timeseries import TimeSeries
from nitime.analysis import FilterAnalyzer

n_voxels = np.prod(data.shape[:-1])

# An example of filtering with a bandpass filter
sub_filtered = np.copy(data)
TR = data_.header.get_zooms()[3]
for v in tqdm(range(15,20)): # some example voxels
    voxel_ts = data[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    T = TimeSeries(voxel_ts.T, sampling_interval=TR)
    F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
    tmp_TC_filt = F.filtered_boxcar.data
    tmp_TC_filt = tmp_TC_filt.T
    sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt
    fig,axs = plt.subplots(1,2, figsize=(9,3))
    axs[0].plot(tmp_TC_filt, alpha=1)
    axs[0].set_title('filtered')
    axs[1].plot(voxel_ts, alpha=1)
    axs[1].set_title('unfiltered')

# Introduction to EEG data in python

In [ ]:
! pip install mne

In [ ]:
import mne

In [ ]:
files_dict = {'eeg_vhdr' : ('subj_0001.vhdr','1oKyrq9a4ylCDo4JRNb_yANVoc-EUOYhe'),
              'eeg_vmrk':   ('subj_0001.vmrk','1VPT5SrDvDmTbCfsiGzKaZptIX-nL701g'),
              'eeg_eeg': ('subj_0001.eeg', '1m6p8D5o6POFm8WasqI_3_5h03T4LI0gx')}

In [ ]:
for key in files_dict.keys():
  download_from_drive(key)

In [ ]:
# Path to the .vhdr file (make sure all the three files are in the same directory)


file_path = '/content/subj_0001.vhdr' # The .vhdr file points to the others

# Load the EEG data from the .vhdr file
raw = mne.io.read_raw_brainvision(file_path, preload=True)

# Print general informations about the loaded data
print(raw.info)

# Plot the raw EEG data
raw.plot(n_channels=62);

In [ ]:
sfreq = raw.info['sfreq']
print(f"Sampling frequency: {sfreq} Hz")

In [ ]:
# Plot the names of all the electrodes
for ch_name, ch_type in zip(raw.info['ch_names'], raw.get_channel_types()):
    print(f"Channel: {ch_name}, Type: {ch_type}")

In [ ]:
# Plot the montage, i.e. the location of the channels over the scalp
raw.plot_sensors(kind='topomap', sphere=(0, 0, 0, 0.12));
# raw.plot_sensors(kind='topomap');

In [ ]:
# extract the data as a numpy array (channels x samples)
data, times = raw.get_data(return_times=True) # times in seconds

In [ ]:
data.shape

In [ ]:
# Get the channels position
pos = mne.channels.layout._find_topomap_coords(raw.info, picks='eeg')

In [ ]:
# Pick a time point, e.g., 1 seconds
time_point = 1.0  # in seconds

# Find index for the chosen time
idx = (np.abs(times - time_point)).argmin()

# Extract values for that time point
values = data[:, idx]

In [ ]:
values.shape

In [ ]:
mne.viz.plot_topomap(values,
                     pos,
                     names=raw.info['ch_names'],
                     sphere=(0, 0, 0, 0.12),
                     size=4);

In [ ]:
# Some basic preprocessing - bandpass filtering
raw_filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0)


In [ ]:
raw_filtered.plot(n_channels=62);

In [ ]:
# Some basic preprocessing - ICA
ica = mne.preprocessing.ICA(n_components=20, random_state=42, max_iter='auto')
ica.fit(raw_filtered)


In [ ]:
ica.plot_components(show_names=True);

In [ ]:
ica.plot_sources(raw_filtered, picks=[i for i in range(15)]);
ica.plot_sources(raw_filtered, picks=[i for i in range(15,30)]);

In [ ]:
### Print Power spectrum of ICA decomposition ###
sources = ica.get_sources(raw_filtered)

# 5. Plot the power spectrum for each ICA component
n_components = ica.n_components_

# Loop through each ICA component
for i in range(n_components):
    # Extract the signal of component i
    component_data = sources.get_data(picks=[i])  # Get data for the i-th ICA component

    # Compute the power spectral density (PSD) of the component
    psd, freqs = mne.time_frequency.psd_array_welch(
        component_data[0],  # Extract the first row (since it's a single component)
        sfreq=raw.info['sfreq'],  # Sampling frequency from the raw data
        fmin=1, fmax=40,  # Focus on the 1-40 Hz range
        n_fft=2048  # Length of FFT (controls frequency resolution)
    )

    # Plot the power spectrum of the component
    plt.figure(figsize=(5, 3))
    plt.plot(freqs, 10 * np.log10(psd), label=f'Component {i}')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Power (dB)')
    plt.title(f'Power Spectrum of ICA Component {i}')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()